# 03 · Caching, performance, and reproducibility

Caching is the highest-leverage performance feature in Flyte: a cache hit costs milliseconds,
while even a trivial task execution costs pod-scheduling seconds. This notebook covers the
v2 cache model end-to-end, then a performance clinic on the most common bottlenecks.

**Learning goals**

1. Use every `Cache` behavior deliberately: `auto`, `override`, `disable`
2. Shape cache keys: `ignored_inputs`, `salt`, custom policies
3. Prevent duplicate concurrent work with `serialize=True`
4. Understand deterministic builds and when to pin `version_override`
5. Diagnose queueing delays, image-pull overhead, and use spot capacity safely

In [ ]:
import flyte
from flyte import Cache

flyte.init_from_config()

env = flyte.TaskEnvironment(
    name="caching_patterns",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)

## 1. Cache behaviors

A task's cache key = **inputs + task version**. The `behavior` decides how that version is derived:

| Behavior | Version comes from | Use for |
|---|---|---|
| `"auto"` | Hash of the task's code — edits invalidate automatically | Development, most pipelines |
| `"override"` | A string **you** pin (`version_override`) | Production stability across code refactors |
| `"disable"` | — (never cached) | Side-effectful tasks (DB writes, notifications) |

In [ ]:
@env.task(cache="auto")                    # shorthand for Cache(behavior="auto")
async def expensive_transform(data: str) -> str:
    print("cache MISS — executing")
    return data.upper()


run = await flyte.run.aio(expensive_transform, data="hello")
await run.wait.aio()

# Run it again with the same input: this one is a cache hit and completes ~instantly.
run2 = await flyte.run.aio(expensive_transform, data="hello")
await run2.wait.aio()
print(run2.url)

In [ ]:
# Production: survive refactors, invalidate only when YOU bump the version
@env.task(cache=Cache(behavior="override", version_override="v3"))
async def stable_transform(data: str) -> str:
    return data.lower()


# Side effects: never cache
@env.task(cache=Cache(behavior="disable"))
async def notify(message: str) -> None:
    print(f"sending: {message}")

## 2. Shaping the cache key

**`ignored_inputs`** — exclude noise (run IDs, seeds, timestamps) so semantically-identical
calls hit the cache:

In [ ]:
@env.task(cache=Cache(behavior="auto", ignored_inputs="trace_id"))
async def enrich(data: str, trace_id: str) -> str:
    return f"{data}-enriched"


# Different trace_id, same data → cache hit

In [ ]:
# salt: namespace the cache, e.g. per experiment — bump the salt to force a clean slate
@env.task(cache=Cache(behavior="auto", salt="experiment_q3"))
async def featurize(data: str) -> str:
    return f"features({data})"


# serialize=True: if two runs request the same (task, inputs) simultaneously,
# ONE executes and the other waits for its result. Essential for expensive
# training tasks triggered from multiple places.
@env.task(cache=Cache(behavior="auto", serialize=True))
async def train_base_model(dataset_version: str) -> str:
    return f"model-for-{dataset_version}"

**Custom cache policies** let the version depend on something external — e.g. a dataset
registry — so cache invalidation follows your *data*, not your code:

```python
from flyte._cache import CachePolicy

class DatasetVersionPolicy(CachePolicy):
    def get_version(self, salt: str, params) -> str:
        return f"{salt}_{lookup_dataset_version()}"

@env.task(cache=Cache(behavior="auto", policies=[DatasetVersionPolicy()]))
async def dataset_dependent(data_path: str) -> str: ...
```

**Force a re-execution** without touching any definitions (e.g. upstream data corrected):

In [ ]:
run = flyte.with_runcontext(overwrite_cache=True).run(expensive_transform, data="hello")
print(run.url)

## 3. Deterministic builds and reproducibility

Three layers decide whether "the same pipeline" really is the same:

1. **Image** — `flyte.Image` builds are content-addressed: same definition → same image.
   Pin every package version in `with_pip_packages(...)` (as these notebooks do); an
   unpinned `"pandas"` resolves differently over time and silently changes your runtime.
2. **Task version** — `cache="auto"` re-versions on *any* code change, including comments.
   For production pipelines pin `Cache(behavior="override", version_override="vN")` and
   treat bumping `vN` as a deliberate, reviewable act.
3. **Inputs** — files/dataframes are content-addressed via object storage (GCS here);
   two runs on the same inputs are byte-identical from Flyte's perspective.

**Recommended lifecycle:** `cache="auto"` while developing → switch hot production tasks to
`override` with an explicit version at release time. That's also the cleanest answer to
"why did yesterday's run cache-miss?" — with `override`, it can't, until you bump it.

## 4. Performance clinic

Where the time actually goes in a task execution, and the lever for each:

| Stage | Symptom | Lever |
|---|---|---|
| **Queueing** | Run pending, no pod | `Timeout(max_queued_time=...)` to fail fast; node-pool capacity (platform team); `queue=` targeting |
| **Image pull** | Pod `ContainerCreating` for minutes | Smaller images; reusable containers ([04](./04-reusable-containers.ipynb)); pre-pulled node images |
| **Cold start** | Every tiny task costs ~10-60s | Reusable containers ([04](./04-reusable-containers.ipynb)) |
| **Execution** | Task slow | Right-size resources; fan out ([02](./02-parallelism-and-resilience.ipynb)) |
| **I/O** | Slow start/end of task | Keep large data in `File`/`Dir` (streamed from GCS, [05](./05-gcp-data-and-integrations.ipynb)), not in inline inputs/outputs |

Two cluster-level levers you control *from code*:

> 💬 **Discuss:** which of your production tasks should move from `auto` to `override`, and what would your team's process be for bumping the version — PR review? release checklist?

In [ ]:
# queue: target a specific node pool / scheduling queue your platform team exposes
# interruptible: allow spot/preemptible nodes — big savings for retry-tolerant work
spot_env = flyte.TaskEnvironment(
    name="spot_batch",
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    interruptible=True,          # spot eviction does NOT consume your retry budget
    # queue="batch-pool",        # uncomment with the queue name for your cluster
)


@spot_env.task(retries=3)
async def preemptible_step(shard: str) -> str:
    return f"processed {shard}"


# Critical steps opt back out at the task level:
@spot_env.task(interruptible=False)
async def checkpoint_results(data: str) -> str:
    return f"persisted {data}"

### At the Kubernetes level

`interruptible=True` schedules onto your spot node pool (taint/toleration wiring is in the
Helm values — appendix A → *Node pools*). GKE gives spot VMs a ~30s termination notice;
Flyte reschedules the attempt on eviction without burning user retries. Pair spot workers
with `@flyte.trace` checkpoints ([02](./02-parallelism-and-resilience.ipynb) §5) so long tasks
resume mid-flight instead of restarting.

## Further reading

- Next: [04-reusable-containers](./04-reusable-containers.ipynb) — kills the cold-start row
  of the table above